# Part 4 — Cache core walkthrough (SQLite + deterministic keys)

This notebook is a guided tour of `src/oqe/cache.py`.

What you'll do:
1. Understand canonicalization + deterministic cache keys
2. See `asof` normalization behavior
3. Create a SQLite cache, write/read entries, and observe TTL expiry
4. Peek inside the SQLite table

> Tip: run this notebook via Poetry so imports work:
```bash
poetry run jupyter lab
# or
poetry run jupyter notebook
```


## 0) Import helpers

We’ll try to import `oqe.cache`. If you're not running from the repo root (or the package isn't installed), we’ll add `./src` to `sys.path`.


In [ ]:
import sys
from pathlib import Path

repo_root = Path().resolve()
src_path = repo_root / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

Imported oqe.cache OK
ORDER_INSENSITIVE_FIELDS: {'option_symbols', 'symbols', 'tickers'}


In [ ]:
from oqe.cache import (
    ORDER_INSENSITIVE_FIELDS,
    SQLiteCache,
    canonical_json,
    make_cache_key,
    normalize_asof,
)

print("Imported oqe.cache OK")
print("ORDER_INSENSITIVE_FIELDS:", ORDER_INSENSITIVE_FIELDS)

## 1) Canonicalization: why it exists

We want a deterministic key for `(tool_name + inputs + asof)`.

But user/tool inputs can be *logically identical* while being *structurally different*:
- dict key order
- list order (sometimes irrelevant, e.g., `symbols=[A,B]` vs `[B,A]`)
- `None` values

`canonical_json(...)` converts inputs into a stable JSON string:
- dict keys are sorted
- `None` values are dropped
- certain lists (field-name based) are treated as order-insensitive and sorted


In [ ]:
inputs_a = {
    "foo": 1,
    "option_symbols": ["O:NVDA251219C00100000", "O:NVDA251219P00100000"],
    "bar": None,
}

inputs_b = {
    "option_symbols": ["O:NVDA251219P00100000", "O:NVDA251219C00100000"],
    "foo": 1,
    "bar": None,
}

canon_a = canonical_json(inputs_a)
canon_b = canonical_json(inputs_b)

print("canon_a:", canon_a)
print("canon_b:", canon_b)
print("canon_a == canon_b ?", canon_a == canon_b)

canon_a: {"foo":1,"option_symbols":["O:NVDA251219C00100000","O:NVDA251219P00100000"]}
canon_b: {"foo":1,"option_symbols":["O:NVDA251219C00100000","O:NVDA251219P00100000"]}
canon_a == canon_b ? True


### Try it yourself

Change the field name from `option_symbols` to something else (e.g., `symbols2`) and re-run.
You should see canonicalization remains deterministic, but the list may stop being treated as order-insensitive.


In [ ]:
inputs_c = {"foo": 1, "symbols2": ["B", "A"]}
inputs_d = {"symbols2": ["A", "B"], "foo": 1}

print("canon_c:", canonical_json(inputs_c))
print("canon_d:", canonical_json(inputs_d))
print("canon_c == canon_d ?", canonical_json(inputs_c) == canonical_json(inputs_d))

canon_c: {"foo":1,"symbols2":["B","A"]}
canon_d: {"foo":1,"symbols2":["A","B"]}
canon_c == canon_d ? False


## 2) `asof` normalization

`normalize_asof(asof)` makes the as-of value stable for keying.

v1 behavior:
- `None` → `None` (later treated as `'latest'`)
- `int/float` → `epoch_ms:<milliseconds>`
- other values → stripped string


In [ ]:
print(normalize_asof(None))
print(normalize_asof(1700000000))  # epoch seconds
print(normalize_asof(1700000000.1234))  # float epoch seconds
print(normalize_asof(" 2025-12-26T10:00:00Z "))

None
epoch_ms:1700000000000
epoch_ms:1700000000123
2025-12-26T10:00:00Z


## 3) Deterministic cache key: `make_cache_key(tool_name, inputs, asof)`

This returns:
- `key` (sha256)
- `asof_norm` (`'latest'` or normalized string)
- `inputs_json` (canonical JSON)


In [ ]:
key1, asof1, canon1 = make_cache_key(
    "get_option_quotes",
    {"option_symbols": ["A", "B"], "foo": 1},
    asof=None,
)
key2, asof2, canon2 = make_cache_key(
    "get_option_quotes",
    {"option_symbols": ["B", "A"], "foo": 1},
    asof=None,
)

print("asof1:", asof1)
print("canon1:", canon1)
print("key1 == key2 ?", key1 == key2)
print("key1:", key1)

asof1: latest
canon1: {"foo":1,"option_symbols":["A","B"]}
key1 == key2 ? True
key1: 5d1316e5c16c998a245d6c00a1758af5c1f21b40f01e04283e65ca01a323c205


## 4) SQLiteCache: write + read

We’ll create a temp sqlite DB, insert a record, then read it back.


In [ ]:
import json
import tempfile

tmpdir = tempfile.TemporaryDirectory()
cache_path = Path(tmpdir.name) / "cache.sqlite"

cache = SQLiteCache(cache_path)
print("Cache DB:", cache_path)

tool = "get_underlying_snapshot"
inputs = {"ticker": "NVDA"}
key, asof_norm, inputs_json = make_cache_key(tool, inputs, asof=None)

payload = {"ticker": "NVDA", "price": 123.45, "note": "example payload"}
response_json = json.dumps(payload, separators=(",", ":"), sort_keys=True)

rec = cache.set(
    key=key,
    tool=tool,
    asof=asof_norm,
    inputs_json=inputs_json,
    response_json=response_json,
    ttl_seconds=60,
)
print("Wrote record:\n", rec)

hit = cache.get(key)
print("\nRead record (hit):\n", hit)
print("\nDecoded payload:", json.loads(hit.response_json) if hit else None)

Cache DB: /var/folders/g_/_gbrp9t921v56m675w45dsjm0000gn/T/tmpd1kw4b3u/cache.sqlite
Wrote record:
 CacheRecord(key='aa103ab21e7ea63df6a4c4ae26dbe48e84dc61e016e38ebe86ff92eba88614de', tool='get_underlying_snapshot', asof='latest', inputs_json='{"ticker":"NVDA"}', response_json='{"note":"example payload","price":123.45,"ticker":"NVDA"}', created_at=1766733911.3950899, ttl_seconds=60, expires_at=1766733971.3950899)

Read record (hit):
 CacheRecord(key='aa103ab21e7ea63df6a4c4ae26dbe48e84dc61e016e38ebe86ff92eba88614de', tool='get_underlying_snapshot', asof='latest', inputs_json='{"ticker":"NVDA"}', response_json='{"note":"example payload","price":123.45,"ticker":"NVDA"}', created_at=1766733911.3950899, ttl_seconds=60, expires_at=1766733971.3950899)

Decoded payload: {'note': 'example payload', 'price': 123.45, 'ticker': 'NVDA'}


## 5) TTL expiry behavior (fake clock)

`SQLiteCache` accepts `now_fn` so expiry is testable/deterministic.


In [ ]:
t = {"now": 1000.0}


def now_fn():
    return t["now"]


tmpdir2 = tempfile.TemporaryDirectory()
cache_path2 = Path(tmpdir2.name) / "cache.sqlite"

cache2 = SQLiteCache(cache_path2, now_fn=now_fn)

tool = "get_underlying_snapshot"
inputs = {"ticker": "SPY"}
key, asof_norm, inputs_json = make_cache_key(tool, inputs, asof=None)

cache2.set(
    key=key,
    tool=tool,
    asof=asof_norm,
    inputs_json=inputs_json,
    response_json='{"ok":true}',
    ttl_seconds=10,
)

print("t=1000 hit?", cache2.get(key) is not None)
t["now"] = 1009.0
print("t=1009 hit?", cache2.get(key) is not None)
t["now"] = 1011.0
print("t=1011 hit?", cache2.get(key) is not None)

cache2.close()

t=1000 hit? True
t=1009 hit? True
t=1011 hit? False


## 6) Peek inside the SQLite table

Inspect raw rows (useful when debugging keys / TTL / JSON payloads).


In [ ]:
import sqlite3

conn = sqlite3.connect(cache_path)
rows = conn.execute(
    "SELECT key, tool, asof, created_at, ttl_seconds, expires_at, inputs_json, response_json FROM cache_entries"
).fetchall()
conn.close()

print("Rows:", len(rows))
if rows:
    row = rows[0]
    print("\nkey:", row[0])
    print("tool:", row[1])
    print("asof:", row[2])
    print("created_at:", row[3])
    print("ttl_seconds:", row[4])
    print("expires_at:", row[5])
    print("inputs_json:", row[6])
    print("response_json:", row[7])

Rows: 1

key: aa103ab21e7ea63df6a4c4ae26dbe48e84dc61e016e38ebe86ff92eba88614de
tool: get_underlying_snapshot
asof: latest
created_at: 1766733911.3950899
ttl_seconds: 60
expires_at: 1766733971.3950899
inputs_json: {"ticker":"NVDA"}
response_json: {"note":"example payload","price":123.45,"ticker":"NVDA"}


## 7) Cleanup

Temporary directories are cleaned up when Python exits, but we can explicitly clean them now.


In [ ]:
cache.close()
tmpdir.cleanup()
tmpdir2.cleanup()
print("Cleaned up temp caches")

Cleaned up temp caches
